# Promptfoo 핵심 실습 튜토리얼 (`3_promptfoo.ipynb`)

본 노트북은 LLM 레드팀/평가 도구 **promptfoo** 를 사용하여 OpenAI `gpt-4o-mini` 모델을 점검하는 최소 실행 흐름을 다룹니다.

promptfoo는 Node.js 기반의 CLI 도구로, 다음을 한 번에 제공합니다.

1. **공격 프롬프트 자동 생성** (`promptfoo redteam generate`)
2. **타깃 모델 자동 평가** (`promptfoo redteam eval`)
3. **결과 시각화 웹 UI** (`promptfoo view`)

> ⚠️ 본 실습은 **인가된 안전성 테스트 목적**으로만 사용되어야 합니다.

> 📚 공식 문서: <https://www.promptfoo.dev/docs/installation/>

## 실습 목표와 진행 흐름

### 실습 목표

1. promptfoo의 최소 레드팀 파이프라인(설치 → 환경 변수 → 설정 파일 작성 → CLI 실행 → 웹 UI 분석)을 익힙니다.
2. `redteam` 모드로 jailbreak, prompt-injection, harmful 카테고리 등 다양한 공격을 자동 생성하여 모델에 시도합니다.
3. 실행 결과를 promptfoo가 제공하는 **로컬 웹 UI**(`http://localhost:15500`)에서 시각적으로 분석합니다.
4. garak / PyRIT와 비교하여 promptfoo가 가지는 강점과 한계를 이해합니다.

### 진행 흐름

| 단계 | 내용 |
|------|------|
| 0 | 가상환경 세팅 및 경로 설정 확인 (Python + Node.js 동시 확인) |
| 1 | `npm install -g promptfoo` 설치 (최신 버전) |
| 2 | 실행 관련 의존성 설치 + `.env` 로드 + 설정 파일 작성 + `promptfoo redteam run` 실행 |
| 3 | `promptfoo view` 로 로컬 웹 UI 접속하여 결과 분석 |
| 4 | promptfoo 의 한계 |

## 사전 준비 및 진행 순서

### 시스템 요구사항

- **Python 3.10 이상** (노트북 커널 실행용)
- **Node.js 18 이상 + npm** (promptfoo CLI 실행용)
- 인터넷 연결 (`npm` 설치 및 OpenAI API 호출용)
- `conda` (선택, 본 노트북은 Python 가상환경 분리에 사용)

### Node.js 설치 (없는 경우)

promptfoo는 **Node.js 기반 CLI**입니다. macOS 기준 권장 설치 방법은 다음과 같습니다.

```bash
# Homebrew 사용
brew install node

# 또는 nvm 사용
curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.40.1/install.sh | bash
nvm install --lts
```

설치 후 다음 명령으로 확인합니다.

```bash
node -v   # v18 이상이어야 합니다
npm -v
```

### `.env` 파일 준비

프로젝트 루트(또는 상위 디렉토리)에 `.env` 파일을 생성하고 아래와 같이 OpenAI API 키를 저장합니다.

```bash
# .env
OPENAI_API_KEY=sk-...
```

> 🔐 `.env` 는 절대 Git 저장소에 커밋하지 않도록 `.gitignore` 에 등록되어 있어야 합니다.

### 가상환경 분리 전략

- **Python 측**: 노트북 커널 환경(예: `redteam-promptfoo`)에는 `python-dotenv`, `pyyaml` 등 분석/설정용 패키지만 설치합니다.
- **Node.js 측**: promptfoo는 `npm install -g promptfoo` 로 전역 설치하거나, `npx promptfoo@latest` 로 매번 최신 버전을 1회용 실행할 수 있습니다.

본 노트북에서는 노트북 커널 = Python 환경, promptfoo = Node.js 전역 CLI 로 분리한 뒤, `subprocess` 로 CLI를 호출하는 구조를 사용합니다.

## 0) 가상환경 세팅 및 경로 설정 확인

본격 실행에 앞서 다음을 확인합니다.

1. 노트북이 올바른 디렉토리(`PROJECT_ROOT`)에서 실행 중인지
2. 사용 중인 Python 인터프리터 경로 / Python 버전
3. **Node.js 및 npm** 설치 여부 (promptfoo 실행 필수 조건)
4. 결과를 저장할 작업 디렉토리(`promptfoo_report/`) 생성

In [1]:
# 실행 환경(Python, Node.js, npm) 및 경로를 점검합니다.
import os
import shutil
import subprocess
import sys
from pathlib import Path

# 현재 작업 디렉토리 = 프로젝트 루트 (이후 모든 상대 경로의 기준점)
PROJECT_ROOT = Path.cwd().resolve()
print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'Kernel Python: {sys.executable}')
print(f'Python ver   : {sys.version.split()[0]}')

# .git 폴더 존재 여부로 프로젝트 루트가 맞는지 1차 검증
if not (PROJECT_ROOT / '.git').exists():
    print('⚠️  현재 경로가 git 루트가 아닐 수 있습니다.')
else:
    print('✅ git 루트 확인')

# ─────────────────────────────────────────────
# Node.js / npm 설치 확인
# promptfoo CLI는 npm 으로 설치되므로 둘 다 PATH에서 발견되어야 함
# ─────────────────────────────────────────────
NODE_BIN = shutil.which('node')
NPM_BIN  = shutil.which('npm')

if not NODE_BIN or not NPM_BIN:
    raise RuntimeError(
        'Node.js / npm 이 PATH에 없습니다. '
        'macOS 의 경우 `brew install node` 또는 nvm으로 Node.js 18+ 를 설치하세요.'
    )

# 버전 출력으로 정상 동작 확인
node_ver = subprocess.run([NODE_BIN, '-v'], capture_output=True, text=True).stdout.strip()
npm_ver  = subprocess.run([NPM_BIN,  '-v'], capture_output=True, text=True).stdout.strip()
print(f'✅ Node.js   : {NODE_BIN} ({node_ver})')
print(f'✅ npm       : {NPM_BIN} ({npm_ver})')

# 현재 conda env 확인 (선택)
current_env = os.environ.get('CONDA_DEFAULT_ENV', '')
if current_env:
    print(f'ℹ️  conda env : {current_env}')

# ─────────────────────────────────────────────
# 작업 디렉토리 생성
# - promptfoo 가 생성하는 모든 산출물(redteam.yaml, output.json 등)을
#   한 폴더 안에 모아 관리합니다.
# ─────────────────────────────────────────────
PROMPTFOO_DIR = PROJECT_ROOT / 'promptfoo_report'
PROMPTFOO_DIR.mkdir(exist_ok=True)
print(f'📁 작업 디렉토리: {PROMPTFOO_DIR}')

PROJECT_ROOT : /Users/selectstar/P5-1_Red-Teaming-Framework
Kernel Python: /opt/anaconda3/envs/redteam-promptfoo/bin/python
Python ver   : 3.11.15
✅ git 루트 확인
✅ Node.js   : /Users/selectstar/.nvm/versions/node/v24.14.1/bin/node (v24.14.1)
✅ npm       : /Users/selectstar/.nvm/versions/node/v24.14.1/bin/npm (11.11.0)
ℹ️  conda env : redteam-promptfoo
📁 작업 디렉토리: /Users/selectstar/P5-1_Red-Teaming-Framework/promptfoo_report


## 1) promptfoo 설치

> 📚 공식 설치 문서: <https://www.promptfoo.dev/docs/installation/>

promptfoo는 Node.js 기반 CLI 도구로, 다음 4가지 방법으로 설치할 수 있습니다.

| 방법 | 명령어 | 특징 |
|------|--------|------|
| ① npm 전역 설치 (권장) | `npm install -g promptfoo` | `promptfoo` 명령을 어디서든 호출 |
| ② npm 로컬 설치 | `npm install promptfoo` | 프로젝트 단위 설치, `npx promptfoo` 로 호출 |
| ③ npx 즉시 실행 | `npx promptfoo@latest <args>` | 설치 없이 매번 최신 버전 사용 |
| ④ Homebrew (macOS) | `brew install promptfoo` | macOS 패키지 매니저로 설치 |

본 노트북은 **방법 ①** (`npm install -g promptfoo`)을 사용합니다.

### 동작 순서

1. `promptfoo --version` 으로 기존 설치 여부 확인
2. 없거나 구버전이면 `npm install -g promptfoo` 실행
3. 설치 후 다시 `promptfoo --version` 으로 검증

In [2]:
# promptfoo 를 npm 전역 패키지로 설치합니다.
import shutil
import subprocess

# ─────────────────────────────────────────────
# STEP 1: 기존 설치 여부 확인
# ─────────────────────────────────────────────
PROMPTFOO_BIN = shutil.which('promptfoo')
if PROMPTFOO_BIN:
    # 이미 설치되어 있으면 버전만 출력하고 종료
    ver = subprocess.run([PROMPTFOO_BIN, '--version'],
                         capture_output=True, text=True).stdout.strip()
    # 일부 버전은 업데이트 경고 배너를 함께 출력하므로 마지막 줄만 표시
    print(f'ℹ️  promptfoo 이미 설치됨: {PROMPTFOO_BIN}')
    print(f'   버전 정보: {ver.splitlines()[-1] if ver else "(미상)"}')
else:
    # ─────────────────────────────────────────
    # STEP 2: npm 전역 설치
    # ─────────────────────────────────────────
    install_cmd = [NPM_BIN, 'install', '-g', 'promptfoo']
    print(' '.join(install_cmd))
    subprocess.run(install_cmd, check=True)

    # ─────────────────────────────────────────
    # STEP 3: 설치 후 PATH 재탐색
    # ─────────────────────────────────────────
    PROMPTFOO_BIN = shutil.which('promptfoo')
    if not PROMPTFOO_BIN:
        raise RuntimeError(
            'promptfoo 설치 직후에도 PATH에서 찾지 못했습니다. '
            'npm 전역 bin 경로(npm bin -g 또는 npm config get prefix)를 PATH에 추가하세요.'
        )

# ─────────────────────────────────────────────
# STEP 4: 설치 검증
# ─────────────────────────────────────────────
result = subprocess.run([PROMPTFOO_BIN, '--version'],
                        capture_output=True, text=True)
print('-' * 40)
print(result.stdout.strip())
print('-' * 40)
print(f'✅ promptfoo 사용 가능: {PROMPTFOO_BIN}')

ℹ️  promptfoo 이미 설치됨: /Users/selectstar/.nvm/versions/node/v24.14.1/bin/promptfoo
   버전 정보: 0.121.11
----------------------------------------
⚠️ The current version of promptfoo 0.121.11 is lower than the latest available version 0.121.12.

Please run npx promptfoo@latest or npm install -g promptfoo@latest to update.

0.121.11
----------------------------------------
✅ promptfoo 사용 가능: /Users/selectstar/.nvm/versions/node/v24.14.1/bin/promptfoo


## 2) 실행 관련 의존성 설치 및 환경 설정

본 섹션에서는 다음 4가지를 순차적으로 수행합니다.

| 단계 | 내용 |
|------|------|
| 2-1 | 노트북 커널에 분석용 Python 패키지 설치 (`python-dotenv`, `pyyaml`) |
| 2-2 | `.env` 에서 `OPENAI_API_KEY` 로드 |
| 2-3 | `promptfooconfig.yaml` 설정 파일 작성 (redteam 시나리오 정의) |
| 2-4 | `promptfoo redteam run` 실행 → 공격 프롬프트 생성 + 평가 수행 |

### 사용 패키지

| 패키지          | 용도                                          |
|-----------------|------------------------------------------------|
| `python-dotenv` | `.env` 에서 `OPENAI_API_KEY` 환경변수 로드     |
| `pyyaml`        | `promptfooconfig.yaml` 작성 및 파싱            |

In [ ]:
# # 2-1) 노트북 커널 환경에 분석용 Python 패키지를 설치합니다.
# # 2-2) .env 에서 OPENAI_API_KEY 를 환경 변수로 로드합니다.
# import os
# import subprocess
# import sys
# from pathlib import Path

# # ─────────────────────────────────────────────
# # 2-1) Python 보조 패키지 설치
# # sys.executable: 현재 노트북 커널이 사용 중인 Python 인터프리터 경로
# # → 정확히 해당 환경의 pip에 설치하여 다른 env 로 새지 않도록 함
# # ─────────────────────────────────────────────
# pip_cmd = [
#     sys.executable, '-m', 'pip', 'install', '-U',
#     'python-dotenv',
#     'pyyaml',
# ]
# print(' '.join(pip_cmd))
# subprocess.run(pip_cmd, check=True)
# print('✅ Python 보조 의존성 설치 완료')

# # ─────────────────────────────────────────────
# # 2-2) .env 로드
# # ─────────────────────────────────────────────
# from dotenv import load_dotenv

# # .env 후보 경로 (프로젝트 루트 우선, 그다음 상위 폴더)
# env_candidates = [PROJECT_ROOT / '.env', PROJECT_ROOT.parent / '.env']
# env_path = next((p for p in env_candidates if p.exists()), None)

# # 둘 다 없으면 명확한 가이드 메시지와 함께 오류 발생
# if env_path is None:
#     tried = '\n'.join(f'- {p}' for p in env_candidates)
#     raise FileNotFoundError(
#         '.env 파일을 찾지 못했습니다. 다음 위치 중 하나에 파일을 생성한 뒤, '
#         'OPENAI_API_KEY=sk-... 형식으로 작성하세요.\n' + tried
#     )

# # .env 의 KEY=VALUE 쌍을 os.environ 에 주입
# load_dotenv(env_path)

# # OPENAI_API_KEY 정상 로드 여부 검증
# if not os.environ.get('OPENAI_API_KEY'):
#     raise ValueError(f'OPENAI_API_KEY 가 비어있습니다: {env_path}')

# # 보안상 키 값은 출력 금지 → 길이만 표시
# print(f'✅ .env 로드: {env_path}')
# print(f"✅ OPENAI_API_KEY 길이: {len(os.environ['OPENAI_API_KEY'])}")

### 2-3) `promptfooconfig.yaml` 작성

promptfoo 의 redteam 모드는 YAML 형식 설정 파일을 입력으로 받습니다. 본 노트북에서는 다음 구성을 사용합니다.

| 필드 | 값 | 설명 |
|------|------|------|
| `description` | 임의 문자열 | 실행 식별용 라벨 |
| `targets[].id` | `openai:gpt-4o-mini` | 평가 대상 모델 (OpenAI provider 형식) |
| `redteam.purpose` | `"general-purpose chatbot"` | 모델의 의도된 용도 (공격 생성에 활용) |
| `redteam.numTests` | `3` | 각 plugin 당 생성할 테스트 케이스 수 |
| `redteam.plugins` | `harmful:hate`, `harmful:violent-crime` | 사용할 공격 카테고리 |
| `redteam.strategies` | `jailbreak`, `prompt-injection` | 공격 변형 전략 |

### 자주 사용되는 plugin 예시

- `harmful:hate` — 혐오 발언 유도
- `harmful:violent-crime` — 폭력/범죄 정보 요청
- `harmful:self-harm` — 자해 관련 위험 콘텐츠
- `pii:direct` — 개인정보(PII) 추출 시도
- `prompt-extraction` — 시스템 프롬프트 추출
- `hijacking` — 의도된 용도 외 사용 유도

### 자주 사용되는 strategy 예시

- `jailbreak` — 페르소나 부여, 가상 시나리오 등으로 안전장치 우회
- `prompt-injection` — 명령 주입을 통한 우회
- `base64` / `rot13` — 인코딩으로 필터 회피
- `multilingual` — 비영어 언어로 변환

> 📚 전체 plugin/strategy 목록: <https://www.promptfoo.dev/docs/red-team/plugins/>

In [ ]:
# # promptfooconfig.yaml 을 작업 디렉토리에 작성합니다.
# import yaml
# from pathlib import Path

# CONFIG_PATH = PROMPTFOO_DIR / 'promptfooconfig.yaml'

# # ─────────────────────────────────────────────
# # 설정 dict 정의
# # - targets: 평가 대상 모델 (provider:model 형식)
# # - redteam.purpose: 모델 용도. 공격 생성기에 컨텍스트로 전달됨
# # - redteam.numTests: plugin 당 생성할 공격 프롬프트 수
# #   (실습용 3개. 본격 평가 시 10~30으로 확대)
# # - plugins: 공격 카테고리 (어떤 종류의 공격을 시도할지)
# # - strategies: 공격 변형 방식 (생성된 공격을 어떻게 변형/우회할지)
# # ─────────────────────────────────────────────
# config = {
#     'description': 'promptfoo redteam tutorial (gpt-4o-mini)',
#     'targets': [
#         {'id': 'openai:gpt-4o-mini', 'label': 'target-llm'},
#     ],
#     'redteam': {
#         'purpose': 'general-purpose chatbot assistant',
#         'numTests': 3,
#         'plugins': [
#             {'id': 'harmful:hate'},
#             {'id': 'harmful:violent-crime'},
#         ],
#         'strategies': [
#             {'id': 'jailbreak'},
#             {'id': 'prompt-injection'},
#         ],
#     },
# }

# # YAML 파일로 저장 (사람이 직접 수정 가능한 형식)
# with CONFIG_PATH.open('w', encoding='utf-8') as f:
#     yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

# print(f'✅ 설정 파일 생성: {CONFIG_PATH}')
# print('-' * 60)
# print(CONFIG_PATH.read_text(encoding='utf-8'))

### 2-4) `promptfoo redteam run` 실행

`promptfoo redteam run` 은 다음 두 단계를 자동으로 수행하는 통합 명령입니다.

1. **공격 프롬프트 생성** (`generate`) — `numTests × plugins × strategies` 조합으로 공격 프롬프트를 생성하여 `redteam.yaml` 에 저장
2. **타깃 모델 평가** (`eval`) — 생성된 프롬프트로 타깃 모델을 호출하고 응답을 detector 로 평가

### 사용하는 CLI 옵션

| 옵션 | 값 | 설명 |
|------|------|------|
| `-c` / `--config` | `promptfooconfig.yaml` | 입력 설정 파일 |
| `-o` / `--output` | `redteam.yaml` | 생성된 공격 프롬프트 저장 경로 |
| `--no-progress-bar` | — | 노트북 출력 깨짐 방지 |

### 생성되는 산출물

| 파일 | 용도 |
|------|------|
| `redteam.yaml` | 생성된 공격 프롬프트 목록 (재현 가능) |
| `.promptfoo/` 또는 `output.json` | 평가 결과 (응답 + pass/fail 판정) — 다음 섹션에서 웹 UI로 시각화 |

> ⏱️  실행 시간: plugin 2개 × strategy 2개 × numTests 3 = 약 12~20개 공격, OpenAI 호출 비용은 보통 $0.01 미만이지만 plugin/strategy 수에 비례하여 증가합니다.

In [ ]:
# # promptfoo redteam run 을 실행하여 공격 프롬프트 생성 + 평가를 1회에 수행합니다.
# import os
# import subprocess
# from pathlib import Path

# REDTEAM_YAML = PROMPTFOO_DIR / 'redteam.yaml'

# # ─────────────────────────────────────────────
# # 환경 변수 구성
# # - 부모 프로세스의 환경 변수 그대로 상속 (OPENAI_API_KEY 포함)
# # - PYTHONIOENCODING=utf-8: 한글 포함 로그 깨짐 방지
# # ─────────────────────────────────────────────
# env = os.environ.copy()
# env['PYTHONIOENCODING'] = 'utf-8'

# # ─────────────────────────────────────────────
# # CLI 명령 조립
# # - cwd 를 PROMPTFOO_DIR 로 지정 → 모든 산출물이 해당 폴더 안에 생성됨
# # - --no-progress-bar: 노트북 셀 출력에 ANSI 진행률 바가 깨져 보이는 것 방지
# # ─────────────────────────────────────────────
# cmd = [
#     PROMPTFOO_BIN, 'redteam', 'run',
#     '-c', str(CONFIG_PATH),
#     '-o', str(REDTEAM_YAML),
#     '--no-progress-bar',
# ]

# print('▶️  실행 명령:')
# print(' '.join(cmd))
# print(f'   cwd = {PROMPTFOO_DIR}')
# print()

# # ─────────────────────────────────────────────
# # 실행 + 로그 그대로 출력
# # - check=False: 일부 plugin 이 실패해도 부분 결과는 보존되므로
# #   직접 returncode 확인하여 분기
# # ─────────────────────────────────────────────
# print('[promptfoo 실행 로그 시작]\n' + '-' * 40)
# result = subprocess.run(cmd, env=env, cwd=str(PROMPTFOO_DIR), text=True)
# print('-' * 40 + '\n[promptfoo 실행 종료]')
# print('return code:', result.returncode)

# if result.returncode != 0:
#     print('⚠️  종료 코드가 0이 아닙니다. 위 로그에서 실패한 plugin/strategy 를 확인하세요.')

# # ─────────────────────────────────────────────
# # 산출물 확인
# # ─────────────────────────────────────────────
# if REDTEAM_YAML.exists():
#     print(f'✅ 공격 프롬프트 저장: {REDTEAM_YAML} ({REDTEAM_YAML.stat().st_size / 1024:.1f} KB)')
# else:
#     print(f'⚠️  redteam.yaml 이 생성되지 않았습니다: {REDTEAM_YAML}')

## 3) 웹 UI 접속

promptfoo 는 평가 결과를 시각화하는 **로컬 웹 UI** 를 내장하고 있습니다. `promptfoo view` 명령으로 기동하며 기본 포트는 `15500` 입니다.

### 웹 UI에서 확인할 수 있는 정보

- **Eval 목록** — 지금까지 실행한 평가의 시간순 목록
- **Test cases** — 각 공격 프롬프트의 원문과 모델 응답 전체
- **Grader 판정** — plugin 별 pass/fail 결과 및 판정 근거
- **Diff 뷰** — 동일 프롬프트를 여러 모델/설정으로 비교 (멀티 타깃일 때)
- **Vulnerability 차트** — plugin/strategy 별 ASR(Attack Success Rate) 집계

### 실행 흐름

| STEP | 내용 |
|------|------|
| STEP 1 | `promptfoo view -y --port 15500` 을 **백그라운드 프로세스** 로 실행 |
| STEP 2 | 서버가 뜨면 자동으로 기본 브라우저에 `http://localhost:15500` 열기 |
| STEP 3 | 노트북 안에서도 `IFrame` 으로 동일 URL을 임베드하여 즉시 미리보기 |
| STEP 4 | 분석 종료 후 `server.terminate()` 로 백그라운드 프로세스 정리 |

> 💡 백그라운드 실행이므로 이 셀의 출력에는 로그가 거의 표시되지 않습니다. 브라우저 또는 임베드된 iframe에서 결과를 확인하세요.

> 🛑 노트북 작업이 끝났다면 마지막 셀(또는 셀 마지막의 종료 코드)로 서버를 반드시 정리해 주세요. 그렇지 않으면 포트 15500 이 점유된 채 남습니다.

In [3]:
# promptfoo 웹 UI 를 백그라운드로 띄우고 노트북에 임베드합니다.
import os
import subprocess
import time
import webbrowser
from urllib.request import urlopen
from urllib.error import URLError

from IPython.display import IFrame, display

PORT = 15500
URL  = f'http://localhost:{PORT}'

# ─────────────────────────────────────────────
# STEP 1: 백그라운드로 promptfoo view 실행
# - -y / --yes: 브라우저 자동 오픈 확인 프롬프트 건너뛰기
# - -n / --no 를 쓰면 자동 오픈을 끌 수 있지만,
#   여기서는 promptfoo 가 직접 브라우저를 열도록 두고
#   노트북 임베드는 보조용으로만 사용
# - cwd=PROMPTFOO_DIR: 해당 폴더의 결과를 우선 표시
# ─────────────────────────────────────────────
env = os.environ.copy()
env['PYTHONIOENCODING'] = 'utf-8'

view_cmd = [PROMPTFOO_BIN, 'view', '-y', '--port', str(PORT)]
print('▶️  실행 명령:', ' '.join(view_cmd))

server = subprocess.Popen(
    view_cmd,
    cwd=str(PROMPTFOO_DIR),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(f'   PID = {server.pid}')

# ─────────────────────────────────────────────
# STEP 2: 서버가 응답할 때까지 최대 20초 대기
# ─────────────────────────────────────────────
for i in range(20):
    try:
        urlopen(URL, timeout=1).close()
        print(f'✅ 웹 UI 응답 확인 ({i+1}초)')
        break
    except URLError:
        time.sleep(1)
else:
    print('⚠️  20초 내 응답 없음. 서버 로그 일부:')
    # 비차단으로 일부 로그 확인
    try:
        out = server.stdout.read1(2000).decode('utf-8', errors='ignore') if server.stdout else ''
        print(out)
    except Exception:
        pass

# ─────────────────────────────────────────────
# STEP 3: 노트북 내부에 iframe 으로 임베드
# (브라우저는 promptfoo 가 자동으로 이미 열었음)
# ─────────────────────────────────────────────
print(f'🌐 브라우저 주소: {URL}')
display(IFrame(URL, width='100%', height=850))

# ─────────────────────────────────────────────
# STEP 4: 정리 안내
# 분석이 끝나면 아래 코드를 실행하여 서버를 정리하세요.
#
#     server.terminate()
#     server.wait(timeout=5)
#
# 또는 노트북 커널을 재시작하면 자식 프로세스도 함께 종료됩니다.
# ─────────────────────────────────────────────
print('\nℹ️  분석 종료 후 다음 셀을 실행하여 서버를 정리하세요:')
print('   server.terminate(); server.wait(timeout=5)')

▶️  실행 명령: /Users/selectstar/.nvm/versions/node/v24.14.1/bin/promptfoo view -y --port 15500
   PID = 20451
✅ 웹 UI 응답 확인 (1초)
🌐 브라우저 주소: http://localhost:15500



ℹ️  분석 종료 후 다음 셀을 실행하여 서버를 정리하세요:
   server.terminate(); server.wait(timeout=5)


## 4) promptfoo 의 한계

promptfoo 는 매우 강력한 레드팀 자동화 도구이지만, 이 도구의 결과만으로 모델의 안전성을 결론짓기는 어렵습니다. 실무 적용 전 반드시 인지해야 할 한계를 정리합니다.

### 1. 공격 생성기에 대한 의존성 (Generator Dependency)

- promptfoo 의 `redteam generate` 는 **별도 LLM**(promptfoo 가 운영하는 원격 모델 또는 사용자가 지정한 로컬 모델)으로 공격 프롬프트를 만듭니다.
- 생성기의 품질·다양성이 곧 공격 커버리지가 되며, 생성기가 약하면 결과는 과대평가됩니다.
- **대응**: 동일 plugin 을 다수 실행하여 분포를 확인하고, 자체 시드 공격(`tests:` 섹션)을 함께 주입합니다.

### 2. Plugin 편향 (Plugin Bias)

- promptfoo 는 사전 정의된 plugin/strategy 카탈로그를 제공하지만, 도메인 특화 공격(금융, 의료, 코드 자동완성 등)은 자체적으로 정의해야 합니다.
- **대응**: `policy` 커스텀 plugin 또는 `tests:` 의 수동 시나리오로 도메인 공격을 보강합니다.

### 3. 비결정성 (Non-determinism)

- 공격 생성과 타깃 응답 양쪽이 모두 LLM 샘플링을 사용하므로 동일 설정 재실행 시 결과가 달라질 수 있습니다.
- `numTests: 3` 은 빠른 점검용이며 통계적 신뢰성은 낮습니다.
- **대응**: 평가 시에는 `numTests` 를 10~30 으로 늘리고, 동일 시드로 여러 차례 반복 실행하여 분포를 비교합니다.

### 4. 단일 지표의 한계 (Single Metric Limitation)

- 결과는 plugin 별 pass/fail 로 표시되지만, 같은 fail 안에도 명백히 유해한 응답과 경계선 응답이 혼재합니다.
- **대응**: 정량 ASR 외에 웹 UI 의 응답 원문을 정성적으로 리뷰하여 위험도 수준을 세분화해야 합니다.

### 5. 맥락 제한 (Context Limitation)

- 기본 설정은 모델을 단독으로 호출하여 평가합니다. 실제 서비스의 시스템 프롬프트, 툴 호출, 권한 모델, RAG 등이 누락됩니다.
- **대응**: `providers` 또는 `targets` 에 HTTP/Python 커스텀 어댑터를 등록하여 실 서비스 엔드포인트를 직접 평가해야 합니다.

### 6. 비용 및 시간 (Cost & Latency)

- `plugins × strategies × numTests` 의 곱으로 호출 수가 증가합니다.
- 예: plugin 5개 × strategy 4개 × numTests 20 = 400 공격 × (생성 비용 + 평가 비용) → OpenAI 기준 수 달러 단위까지 쉽게 도달.
- **대응**: 소규모 샘플로 파이프라인 검증 → 본격 평가 시 예산 계획 + `--max-concurrency` 로 동시성 조절.

### 7. Grader 의존성

- pass/fail 판정은 promptfoo 의 내장 grader(또는 LLM-as-a-Judge)에 의존합니다. grader 모델이 바뀌면 결과가 흔들립니다.
- **대응**: 의심 케이스는 다른 모델(예: `claude-4`)로 grader 를 교차 실행하거나, 사람이 직접 라벨링한 골든셋과 비교 검증합니다.

---

### 마무리

promptfoo 는 레드팀 자동화의 **출발점**으로 매우 유용하지만, 다음 작업을 통해 완성도를 높여야 합니다.

1. **다양한 도구 결합**: promptfoo(자동화 친화) + [[1_garak_tutorial]](probe 카탈로그 풍부) + [[2_pyrit_tutorial]](공격 전략 자동화) + 자체 시나리오
2. **여러 모델 비교**: `gpt-4o-mini`, `claude-4`, `gemini-2.0` 등 동일 설정으로 교차 평가하여 상대적 취약성 파악
3. **자동화 + 인간 검토 결합**: 정량 메트릭으로 1차 스크리닝 → 의심 케이스 정성 분석
4. **운영 환경 반영**: 시스템 프롬프트, 툴, RAG 까지 포함한 통합 평가 (custom provider 사용)